# Verify: automated output verification before agent actions

Before a Managed Agent acts on its own output—writing files, sending messages,
calling downstream APIs—how do you know its claims are accurate?
Human-in-the-loop (see `CMA_gate_human_in_the_loop.ipynb`) is one answer.
Automated verification is the complement: fast, scalable, and suitable for
high-volume pipelines where human review of every step isn't feasible.

This notebook builds a **verification gate** as a stateless hand.
The pattern:

1. A primary agent produces output (analysis, recommendation, summary).
2. A verification hand independently checks the claims in that output.
3. A trust receipt records what was checked, what passed, and what confidence was assigned.
4. The pipeline proceeds only if the receipt meets a configurable threshold.

## Why a stateless hand, not another agent?

The [Scaling Managed Agents](https://www.anthropic.com/engineering/managed-agents)
post describes hands as stateless functions: `execute(name, input) → string`.
Verification fits this shape perfectly—it takes the primary output as input,
runs an independent check, and returns a structured verdict. No session state,
no tool calls, just a single inference step.

This keeps verification fast (one API call) and auditable
(the receipt is a plain dict you can store alongside the action log).

In [ ]:
import hashlib
import json
import os
from dataclasses import asdict, dataclass
from datetime import datetime, timezone

import anthropic

MODEL = os.environ.get("COOKBOOK_MODEL", "claude-haiku-4-5")
client = anthropic.Anthropic()

## 1. Define the verification receipt schema

A receipt captures exactly what the verifier checked and how confident it was.
Storing receipts alongside action logs gives you a full audit trail.

In [ ]:
@dataclass
class ClaimVerdict:
    claim: str
    supported: bool
    confidence: float  # 0.0-1.0
    note: str


@dataclass
class VerificationReceipt:
    timestamp: str
    primary_output_hash: str  # first 16 hex chars of sha256
    verdicts: list
    overall_confidence: float
    approved: bool
    threshold_used: float

    def display(self) -> None:
        status = "\u2705 APPROVED" if self.approved else "\u274c BLOCKED"
        print(f"{status}  (confidence {self.overall_confidence:.2f} / threshold {self.threshold_used:.2f})")
        for v in self.verdicts:
            mark = "\u2713" if v.supported else "\u2717"
            print(f"  {mark} [{v.confidence:.2f}] {v.claim}")
            if v.note:
                print(f"       \u2192 {v.note}")

## 2. The verification hand

The hand takes the primary agent's output text plus any reference material
and returns a `VerificationReceipt`. It uses a single `client.messages.create`
call to keep it stateless and fast.

In [ ]:
VERIFY_SYSTEM = """\
You are a verification hand. Your job is to check whether the claims in an agent's
output are accurate and supported by the provided context.

For each claim you identify, return a JSON object with this schema:
{
  "verdicts": [
    {
      "claim": "<exact claim text>",
      "supported": true | false,
      "confidence": <0.0 to 1.0>,
      "note": "<brief explanation>"
    }
  ]
}

Be conservative: if a claim cannot be verified from the context, mark it unsupported.
Extract at most 5 key factual claims from the output.
Respond with valid JSON only, no prose."""


def verify_output(
    primary_output: str,
    context: str = "",
    threshold: float = 0.75,
) -> VerificationReceipt:
    """Run a single-inference verification check on primary_output."""
    user_content = f"## Agent output to verify\n\n{primary_output}"
    if context:
        user_content += f"\n\n## Reference context\n\n{context}"

    response = client.messages.create(
        model=MODEL,
        max_tokens=1024,
        system=VERIFY_SYSTEM,
        messages=[{"role": "user", "content": user_content}],
    )

    raw = response.content[0].text.strip()
    # Strip markdown code fences if present
    if raw.startswith("```"):
        raw = raw.split("\n", 1)[1].rsplit("```", 1)[0]
    data = json.loads(raw)

    verdicts = [ClaimVerdict(**v) for v in data["verdicts"]]
    overall = sum(v.confidence for v in verdicts) / len(verdicts) if verdicts else 0.0

    return VerificationReceipt(
        timestamp=datetime.now(timezone.utc).isoformat(),
        primary_output_hash=hashlib.sha256(primary_output.encode()).hexdigest()[:16],
        verdicts=verdicts,
        overall_confidence=overall,
        approved=overall >= threshold,
        threshold_used=threshold,
    )

## 3. Simulated agent outputs

We'll test the gate against two synthetic outputs:
- **Output A**: accurate summary of a well-known Python release with verifiable facts.
- **Output B**: fabricated output with several unsupported claims—typical hallucination risk.

In production, these would come from a Managed Agent session.

In [ ]:
# Simulated primary agent output - mostly accurate
OUTPUT_A = """\
Python 3.12 was released in October 2023. Key improvements include improved error
messages that show the exact location of the problem, a new type parameter syntax
for generic classes and functions (PEP 695), and a 5% performance improvement on
the standard benchmark suite. The @override decorator was added to the standard
library typing module.
"""

CONTEXT_A = """\
Python 3.12.0 release date: 2023-10-02.
PEP 695 - Type Parameter Syntax: accepted and implemented in 3.12.
Performance: CPython 3.12 is approximately 5% faster than 3.11 on pyperformance.
typing.override: Added in Python 3.12, backported via typing_extensions.
Improved error messages with precise locations: yes, introduced in 3.12.
"""

print("=== Output A (mostly accurate) ===")
receipt_a = verify_output(OUTPUT_A, CONTEXT_A, threshold=0.75)
receipt_a.display()

In [ ]:
# Simulated primary agent output - contains hallucinations
OUTPUT_B = """\
Python 3.13 introduced a new just-in-time compiler that delivers 50% speed
improvements on all workloads. The GIL was completely removed in 3.13 for all
Python objects. Additionally, a new built-in match_all() function was added
that works similarly to re.findall but supports fuzzy matching.
"""

CONTEXT_B = """\
Python 3.13: Released October 2024. Experimental JIT added (not production-ready,
not 50% improvement on all workloads - varies by benchmark).
Free-threaded mode: GIL can be disabled with a special build flag, but GIL was
NOT completely removed by default in 3.13.
No built-in match_all() function exists in Python 3.13 or any version.
"""

print("=== Output B (contains hallucinations) ===")
receipt_b = verify_output(OUTPUT_B, CONTEXT_B, threshold=0.75)
receipt_b.display()

## 4. The trust gate

The gate is a simple function that either passes the output to the next step
or raises an exception. You can replace the exception with a human escalation
path (see `CMA_gate_human_in_the_loop.ipynb`) or a retry loop.

In [ ]:
class VerificationError(Exception):
    """Raised when an agent's output fails the verification gate."""

    def __init__(self, receipt: VerificationReceipt):
        self.receipt = receipt
        super().__init__(
            f"Verification failed: confidence {receipt.overall_confidence:.2f} "
            f"< threshold {receipt.threshold_used:.2f}"
        )


def trust_gate(
    output: str,
    context: str = "",
    threshold: float = 0.75,
    action_label: str = "downstream action",
) -> VerificationReceipt:
    """
    Verify output and raise VerificationError if it doesn't meet the threshold.
    Returns the receipt so callers can log it alongside the action.
    """
    receipt = verify_output(output, context, threshold)
    if not receipt.approved:
        raise VerificationError(receipt)
    print(f"Gate passed \u2014 proceeding with: {action_label}")
    return receipt


# Run the gate on both outputs
print("--- Gate on Output A ---")
try:
    receipt = trust_gate(
        OUTPUT_A, CONTEXT_A, threshold=0.75, action_label="publish Python 3.12 summary"
    )
    # In production: log the receipt, then call your downstream action
    print(f"Receipt stored: hash={receipt.primary_output_hash}  ts={receipt.timestamp}")
except VerificationError as e:
    print(f"Blocked: {e}")

print()
print("--- Gate on Output B ---")
try:
    receipt = trust_gate(
        OUTPUT_B, CONTEXT_B, threshold=0.75, action_label="publish Python 3.13 summary"
    )
    print(f"Receipt stored: hash={receipt.primary_output_hash}  ts={receipt.timestamp}")
except VerificationError as e:
    print(f"Blocked: {e}")
    print("Routing to human review or retry loop.")

## 5. Connecting to a Managed Agent session

In a real pipeline, the primary output comes from a CMA session.
Here's how to drop the verification gate into that flow:

```python
session = client.beta.agents.sessions.create(agent_id=AGENT_ID)

# Run the agent to produce output
result = client.beta.agents.sessions.resume(
    session_id=session.id,
    input={"content": user_task},
)
primary_output = result.output_text

# Verify before acting
receipt = trust_gate(
    output=primary_output,
    context=reference_material,       # docs, retrieved chunks, etc.
    threshold=0.80,                    # raise the bar for production
    action_label="write report to GCS",
)

# Log receipt + execute action
log_action(receipt=asdict(receipt), output=primary_output)
write_to_storage(primary_output)
```

The receipt travels with the action log, giving you full traceability:
which claims were checked, at what confidence, and against what context.

### Tuning the threshold

| Use case | Suggested threshold |
|---|---|
| Internal draft / low-stakes summary | 0.60 |
| Customer-facing copy | 0.75 |
| Financial / medical / legal | 0.90 |

A lower threshold lets more outputs through; a higher one catches more
hallucinations at the cost of more false rejections. Pair this gate with
`CMA_gate_human_in_the_loop.ipynb` to route blocked outputs to a reviewer
rather than failing silently.